Клонируем репозиторий, из которого будем брать данные, на совнове которых будем создавать данные для обучение ретривера

In [ ]:
!git clone https://github.com/chipslays/russian-recipes-parser.git ./data/

In [ ]:
!pip install torch
!pip install langchain_text_splitters
!pip install transformers torch accelerate bitsandbytes

In [ ]:
import torch
import glob
import json

from huggingface_hub import login
from tqdm import tqdm
from io import StringIO
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 60
RECIPE_LIST_PATH = "data/russian-recipes-parser/storage/recipes/[0-9][0-9][0-9]/*.json"
ALL_RECIPES_FILENAME = 'data/all_recipes.json'
ALL_RECIPE_CHUNKS_FILENAME = 'data/all_recipe_chunks.json'
print(f"Используем устройство: {DEVICE}")

In [ ]:
COLAB_HUGGING_FACE_ACCESS_TOKEN = "" # your token
login(token=COLAB_HUGGING_FACE_ACCESS_TOKEN)

Загружаем данные из склонированного репозитория

In [ ]:
parsed_recipe_list = []

recipe_file_list = glob.glob(RECIPE_LIST_PATH)
print(f"Найдено {len(recipe_file_list)} файлов рецептов для обработки.")
for step_index, filepath in tqdm(enumerate(recipe_file_list), total=len(recipe_file_list), desc="Обработка рецептов"):
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            recipe_data = json.load(f)
        parsed_recipe = {
            "name": recipe_data.get('title', ''),
            "ingredients": recipe_data.get('ingredients', []),
            "instruction": recipe_data.get('instruction', []),
            "full_text": "",
        }
        full_recipe_text = StringIO()
        recipe_title = f"Название рецепта: {parsed_recipe['name']}\n\n"
        full_recipe_text.write(recipe_title)
        ingredients_data_list = parsed_recipe['ingredients']
        if isinstance(ingredients_data_list, list) and ingredients_data_list:
            ingredients_data = ingredients_data_list[0]
            full_recipe_text.write(f"{ingredients_data.get('name', '')}:\n")
            for step_index, ingredient in enumerate(ingredients_data.get('list', [])):
                full_recipe_text.write(f"{step_index + 1}) {ingredient.get('name', '')} {ingredient.get('value', '')} {ingredient.get('type', '')}\n")
        full_recipe_text.write(f"\nИнструкции по приготовлению:\n")
        for step_index, instruction_data in enumerate(recipe_data.get('instruction', [])):
          full_recipe_text.write(f"{step_index + 1}) {instruction_data.get('text', '')}\n")
        parsed_recipe["full_text"] = full_recipe_text.getvalue()
        parsed_recipe_list.append(parsed_recipe)
    except Exception as e:
        print(f"Ошибка при обработке файла {filepath}: {e}")

print(f"Загружено {len(parsed_recipe_list)} рецептов.")


In [ ]:
with open(ALL_RECIPES_FILENAME, 'w', encoding='utf-8') as f:
    json.dump(parsed_recipe_list, f, ensure_ascii=False, indent=2)

print(f"Сохранено {len(parsed_recipe_list)} рецептов в {ALL_RECIPES_FILENAME}")


In [ ]:
with open(ALL_RECIPES_FILENAME, 'r', encoding='utf-8') as f:
    train_chunk_list = json.load(f)

print(f"Загружено {len(train_chunk_list)} чанков")


In [ ]:
class RecipeChunker:
    def chunk_recipe(self, parsed_recipe, max_chunk_size=512):
      chunk_list = []
      parsed_recipe_name = parsed_recipe['name']

      # Уровень 1: Целый рецепт (глобальный контекст)
      chunk_list.append({
        'recipe_name': parsed_recipe_name,
        'scale': 'full',
        'text': parsed_recipe['full_text'],
      })

      # Уровень 2: Секции (ингредиенты)
      if parsed_recipe['ingredients']:
        ingredient_list = parsed_recipe['ingredients'][0]
        ingredient_text = StringIO()
        ingredient_text.write(f"Ингредиенты для рецепта \"{parsed_recipe_name}\":\n")
        for step_index, ingredient in enumerate(ingredient_list['list']):
          ingredient_text.write(f"{step_index + 1}) {ingredient['name']} {ingredient['value']} {ingredient['type']}\n")
        chunk_list.append({
          'recipe_name': parsed_recipe_name,
          'scale': 'ingredients',
          'text': ingredient_text.getvalue()
        })

      # Уровень 3: Отдельные шаги
      if parsed_recipe['instruction']:
        splitter = RecursiveCharacterTextSplitter(
          chunk_size=max_chunk_size,
          chunk_overlap=30,
          separators=[". ", ", ", " "]
        )
        instruction_text_prefix = f"Инструкции для рецепта \"{parsed_recipe_name}\":\n"
        for step_index, instruction_data in enumerate(parsed_recipe['instruction']):
          step_number = step_index + 1
          instruction_text = StringIO()
          instruction_text.write(instruction_text_prefix)
          instruction_text.write(f"{step_number}) {instruction_data['text']}\n")
          if instruction_text.tell() <= max_chunk_size:
            chunk_list.append({
              'recipe_name': parsed_recipe_name,
              'scale': 'step',
              'step_number': step_number,
              'text': instruction_text.getvalue(),
            })
            continue
          instruction_text_chunk_list = splitter.split_text(instruction_text.getvalue())
          for substep_index, instruction_text_chunk in enumerate(instruction_text_chunk_list):
            chunk_list.append({
              'recipe_name': parsed_recipe_name,
              'scale': 'substep',
              'text': instruction_text_chunk,
              'step_number': step_number,
              'substep_number': substep_index + 1,
            })
      return chunk_list


In [ ]:
recipe_chunker = RecipeChunker()
train_chunk_list = []
for retriever_index, parsed_recipe in tqdm(enumerate(parsed_recipe_list), total=len(parsed_recipe_list), desc="Создание чанков рецептов"):
  train_chunk_list.extend(recipe_chunker.chunk_recipe(parsed_recipe))

print(f"Создано {len(train_chunk_list)} чанков разного масштаба")
for retriever_index in range(5):
  train_chunk = train_chunk_list[retriever_index]
  print(json.dumps(train_chunk, ensure_ascii=False, indent=2))


In [ ]:
with open(ALL_RECIPE_CHUNKS_FILENAME, 'w', encoding='utf-8') as f:
    json.dump(train_chunk_list, f, ensure_ascii=False, indent=2)

print(f"Сохранено {len(train_chunk_list)} чанков в {ALL_RECIPE_CHUNKS_FILENAME}")


In [ ]:
class RecipeQuestionGenerator:
    def __init__(self, model_name="Qwen/Qwen2.5-7B-Instruct"):
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="cuda:0",
            trust_remote_code=True,
            use_cache=True
        )
        self.model.eval()
        self.model = torch.compile(self.model, mode="reduce-overhead")
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token


    def generate_question_list(self, recipe_text: str, question_amount=3) -> list[str]:
        question_prompt = self._create_question_prompt(recipe_text, question_amount)
        tokenizer_output = self.tokenizer(question_prompt, return_tensors="pt", truncation=True, max_length=2048).to(self.model.device)
        with torch.inference_mode():
            model_output = self.model.generate( # type: ignore
                **tokenizer_output,
                max_new_tokens=256,
                temperature=0.7,
                do_sample=True,
                top_p=0.9,
                top_k=50,
                repetition_penalty=1.05
            )
        model_response = self.tokenizer.decode(model_output[0], skip_special_tokens=True)
        return self._extract_question_list(model_response, question_amount)


    def _extract_question_list(self, model_response: str, question_amount: int) -> list[str]:
        question_list = []
        question = ""
        if "Вопрос 1:" in model_response:
            model_response = model_response.split("Вопрос 1:")[-1]
            for index in range(1, question_amount + 1):
                current_marker = f"Вопрос {index}:"
                next_marker = f"Вопрос {index+1}:" if index + 1 <= question_amount else None
                if next_marker and next_marker in model_response:
                    question, model_response = model_response.split(next_marker, 1)
                    question = question.replace(current_marker, "").strip()
                else:
                    question = model_response.replace(current_marker, "").strip()
                    if index == question_amount - 1:
                        question = model_response.strip()
                question_list.append(question)
        question_list = [question.strip() for question in question_list if question and '?' in question]
        return question_list[:question_amount]

    def _create_question_prompt(self, recipe_text: str, question_amount: int = 3) -> str:
        return f"""Ты — ассистент, который помогает обучать систему поиска рецептов. Твоя задача — придумать {question_amount} разнообразных вопросов, на которые отвечает данный текст.

Правила:
1. Вопросы должны быть краткими и естественными (как их мог бы задать пользователь)
2. Каждый вопрос должен быть уникальным и затрагивать разные аспекты текста
3. Используй информацию ТОЛЬКО из текста
4. Не задавай вопросы, требующие знаний вне текста

Текст:
{recipe_text}

Отвечай строго в формате:
Вопрос 1: [первый вопрос]
Вопрос 2: [второй вопрос]
Вопрос 3: [третий вопрос]

Не добавляй ничего кроме вопросов в этом формате."""


In [ ]:
question_generator = RecipeQuestionGenerator()
question_list = []
part_of_chunk_list = train_chunk_list[3000:6000]
for retriever_index, train_chunk in tqdm(enumerate(part_of_chunk_list), total=len(part_of_chunk_list), desc="Генерация вопросов для чанков рецептов"):
  if train_chunk['scale'] != 'full':
    continue
  train_chunk['questions'] = question_generator.generate_question_list(train_chunk['text'])


In [ ]:
output_filename = 'recipe_chunks_with_questions_2.json'
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(part_of_chunk_list, f, ensure_ascii=False, indent=2)

print(f"Сохранено {len(part_of_chunk_list)} чанков в {output_filename}")


In [ ]:
input_filename = 'recipe_chunks_with_questions_1.json'
with open(input_filename, 'r', encoding='utf-8') as f:
    train_chunk_list = json.load(f)

print(f"Загружено {len(train_chunk_list)} чанков")


In [ ]:
input_filename = 'recipe_chunks_with_questions_2.json'
with open(input_filename, 'r', encoding='utf-8') as f:
    eval_chunk_list = json.load(f)
print(f"Загружено {len(eval_chunk_list)} чанков")


In [ ]:
train_pruned_chunk_list = []
for train_chunk in train_chunk_list:
    if 'questions' not in train_chunk:
        continue
    if not train_chunk['questions']:
        continue
    train_pruned_chunk_list.append(train_chunk)

eval_pruned_chunk_list = []
for train_chunk in eval_chunk_list:
    if 'questions' not in train_chunk:
        continue
    if not train_chunk['questions']:
        continue
    eval_pruned_chunk_list.append(train_chunk)


In [ ]:
output_filename = 'train_pruned_recipe_chunks_with_questions_1.json'
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(train_pruned_chunk_list, f, ensure_ascii=False, indent=2)
print(f"Сохранено {len(train_pruned_chunk_list)} чанков в {output_filename}")


In [ ]:
output_filename = 'eval_pruned_recipe_chunks_with_questions_2.json'
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(eval_pruned_chunk_list, f, ensure_ascii=False, indent=2)
print(f"Сохранено {len(eval_pruned_chunk_list)} чанков в {output_filename}")
